# Трек C: OWLv2 на соседних кадрах (dt = ±30)

Считает только OWLv2 и только кадры-соседи, которые сами не целевые: 879 кадров, 5173 кропа `top` ×4 (462 из них для train-строк).
Для целевых кадров детекции уже есть в `ov_owl__v1`. В конце три части склеиваются в один источник `ov_owl__v4` с `coverage: all` (1718 кадров): только такой источник `ts.temporal` использует для `prev/next/med`.

**Перед запуском**

1. Settings: Accelerator `GPU T4 x2`, Internet `Off`.
2. Add Input:
   - датасет соревнования;
   - датасет с кодом репозитория (коммит с `--rows neighbours`, `--shard` и `merge_dets.py` для долей neighbours);
   - датасет с весами `ts-models-ov` (каталог `ov_owl/` с `model.safetensors`); без него ноутбук остановится в первой ячейке;
   - `throddy/ts-artifacts-ov-pseudo` (готовые `dets/ov_owl__v1.parquet` и `.json`).
3. В первой ячейке вписать `TS_GIT_SHA`. Save Version → Save & Run All.

**Выход** в `/kaggle/working/artifacts/dets/`: доли `ov_owl__v2` (0/2, GPU 0) и `ov_owl__v3` (1/2, GPU 1) и склейка `ov_owl__v4` (v1 + v2 + v3, `coverage: all`), все с карточками, формат 8.1. Трек A берёт `ov_owl__v4`.

**Время.** Измерено в прогоне 13.11: OWLv2 0.54 с на кроп (fp16, по два кропа, T4). Оценка: 2580 и 2593 кропа на долю, около 23–24 мин модельного времени на каждой карте, доли идут параллельно; плюс загрузка модели. `MAX_MINUTES` с запасом; при штатной остановке подключить вывод как Input и запустить ещё раз, расчёт продолжится.
Тестовые кадры обрабатываются только автоматически.

In [ ]:
import os
from pathlib import Path

os.environ['TS_GIT_SHA'] = ''   # коммит репозитория, из которого собран архив с кодом
PROMPT_VERSION = 2              # тот же промпт, что у ov_owl__v1
MAX_MINUTES = 120               # штатная остановка каждой доли (оценка расчёта около 24 мин)

os.environ['OV_PROMPT'] = str(PROMPT_VERSION)
os.environ['OV_MAX_MINUTES'] = str(MAX_MINUTES)
os.environ['OV_SOURCES'] = 'ov_owl'   # kaggle_setup: нужны веса только OWLv2
found = sorted(p.parents[2] for p in Path('/kaggle/input').rglob('tracks/ov/run_ov.py'))
assert len(found) == 1, f'Нужен ровно один подключённый датасет с кодом репозитория, найдено: {found}'
assert '--shard' in (found[0] / 'tracks/ov/run_ov.py').read_text(encoding='utf-8'), 'В датасете с кодом старая версия run_ov.py: нет --rows neighbours и --shard'
assert 'neighbours' in (found[0] / 'tracks/ov/merge_dets.py').read_text(encoding='utf-8'), 'В датасете с кодом старая версия merge_dets.py: нет склейки долей neighbours'
os.chdir(found[0])
%run tracks/ov/kaggle_setup.py
weights = Path(os.environ['TS_OV_MODELS']) / 'ov_owl' / 'model.safetensors'
v1 = Path(os.environ['TS_ARTIFACTS']) / 'dets' / 'ov_owl__v1.parquet'
assert weights.is_file(), f'Нет весов OWLv2: {weights}'
assert v1.is_file() and v1.with_suffix('.json').is_file(), 'Нет ov_owl__v1: подключите throddy/ts-artifacts-ov-pseudo'

In [ ]:
%%bash
set -u
run() {  # доля, версия, GPU
  python tracks/ov/run_ov.py --source ov_owl --rows neighbours --shard "$1/2" --version "$2" \
    --prompt-version "$OV_PROMPT" --device "cuda:$3" --batch 2 --fp16 --max-minutes "$OV_MAX_MINUTES"
}
# индекс строк строится один раз до запуска двух процессов, чтобы они не писали его одновременно
python -c "from ts.rows import load_rows_ext; print('rows_ext:', len(load_rows_ext()))"
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  run 0 2 0 > /kaggle/working/owl_shard0.log 2>&1 &
  run 1 3 1 > /kaggle/working/owl_shard1.log 2>&1 &
  wait
else
  run 0 2 0 > /kaggle/working/owl_shard0.log 2>&1
  run 1 3 0 > /kaggle/working/owl_shard1.log 2>&1
fi
tail -n 5 /kaggle/working/owl_shard0.log /kaggle/working/owl_shard1.log


In [ ]:
art = Path(os.environ['TS_ARTIFACTS'])
for path in sorted((art / 'dets').glob('ov_owl__v[23].*')):
    print(f'{path.stat().st_size / 2 ** 20:8.2f} МБ  {path.name}')
!python -m ts.check dets {art}/dets/ov_owl__v2.parquet
!python -m ts.check dets {art}/dets/ov_owl__v3.parquet

if all((art / 'dets' / f'ov_owl__v{v}.parquet').is_file() for v in (2, 3)):
    !python tracks/ov/merge_dets.py ov_owl__v1 ov_owl__v2 ov_owl__v3 --version 4
    !python -m ts.check dets {art}/dets/ov_owl__v4.parquet
else:
    print('Доли не готовы: сохраните версию, подключите её вывод как Input и запустите ноутбук ещё раз.')